# PROT-ON: structure-based detection of designer PROTein interface MutatiONs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CSB-KaracaLab/prot-on/blob/main/PROT-ON.ipynb)

PROT-ON takes the structure of a protein dimer, mutates every interface residue of one chain to the other 19 amino acids, and scores each mutant with [EvoEF1](https://github.com/tommyhuangthu/EvoEF) (bundled) or [FoldX](http://foldxsuite.crg.eu/) (your own licensed copy). It then reports the mutations that are predicted to enrich or deplete binding, filtered by their effect on the stability of the scanned chain.

**How to use this notebook**

1. Run the cells from top to bottom (in Colab: *Runtime → Run all*). With the default settings this runs the example complex.
2. To scan your own dimer, change the settings in step 2 and run steps 2 to 4 again.

The example (532 mutations) takes about a minute on a laptop. Larger interfaces take proportionally longer. The notebook runs on Google Colab, Linux and macOS.

If you use PROT-ON, please cite: Koşaca M, Yılmazbilek İ, Karaca E (2023). PROT-ON: A structure-based detection of designer PROTein interface MutatiONs. *Frontiers in Molecular Biosciences* 10:1063971. [doi:10.3389/fmolb.2023.1063971](https://doi.org/10.3389/fmolb.2023.1063971)

## 1. Set up PROT-ON
Downloads PROT-ON (unless the notebook is already inside a copy of the repository), compiles EvoEF1 and installs the plotting packages.

In [ ]:
#@title Set up PROT-ON { display-mode: "form" }
import os, shutil, subprocess, sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/CSB-KaracaLab/prot-on.git"


def _find_repo():
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "proton.py").is_file() and (folder / "EvoEF1" / "src").is_dir():
            return folder
    return None


REPO = _find_repo()
if REPO is None:
    if not Path("prot-on/proton.py").is_file():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO_URL, "prot-on"], check=True)
    REPO = Path("prot-on").resolve()
os.chdir(REPO)
(REPO / "results").mkdir(exist_ok=True)

# EvoEF1 is shipped as source code and compiled here.
EVOEF = REPO / "EvoEF1" / "EvoEF"
if not EVOEF.is_file():
    if shutil.which("g++") is None:
        raise RuntimeError("A C++ compiler (g++) is needed to build EvoEF1, and none was found.")
    build = subprocess.run("g++ -O3 -ffast-math -w -o EvoEF src/*.cpp", shell=True,
                           cwd=REPO / "EvoEF1", capture_output=True, text=True)
    if build.returncode != 0 or not EVOEF.is_file():
        print(build.stderr[-3000:])
        raise RuntimeError("EvoEF1 could not be compiled.")


def _major(package):
    try:
        return int(version(package).split(".")[0])
    except PackageNotFoundError:
        return None


# PROT-ON saves its figures with plotly 5 and kaleido 0.2; newer kaleido needs a Chrome installation.
wanted = [name for name in ("numpy", "pandas", "py3Dmol") if _major(name) is None]
if _major("plotly") != 5 or _major("kaleido") != 0:
    wanted += ["plotly==5.24.1", "kaleido==0.2.1"]
if wanted:
    print("Installing:", ", ".join(wanted))
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *wanted], check=True)

RUNS = {}                                     # every scan of this session, keyed by structure and chain
print("PROT-ON is ready in", REPO)

## 2. Input and settings

**Structure.** PROT-ON needs a PDB file with exactly two chains. Keep `example-input/complex.pdb` to try the example. To scan your own dimer, tick `upload_pdb` (Colab), or put the path *or the web address* of a PDB file in `pdb_file` — a HADDOCK model can be scanned straight from its result page.

A chain of the file can hold more than one polypeptide. An antibody whose heavy and light chains were both given the chain ID `A` is scanned as one molecule, so the whole paratope is covered in a single run.

**Optional PSSM filter.** To also filter the mutations by evolutionary information, provide the PSSM of the scanned chain as a comma-separated file (one row per residue, one column per amino acid), for example from [POSSUM](https://possum.erc.monash.edu/server.jsp). The rows are paired with the chain's residues **in order**, so a chain whose numbering has a gap will be mis-paired: renumber it first. For the example complex, use `example-input/complex_chain_D_pssm.csv`.

In [ ]:
#@title Optional: structures of the HADDOCK antibody–antigen tutorial { display-mode: "form" }
download_tutorial_structures = False #@param {type:"boolean"}
#@markdown The interleukin-1β / gevokizumab complex of the
#@markdown [HADDOCK2.4 antibody–antigen tutorial](https://www.bonvinlab.org/education/HADDOCK24/HADDOCK24-antibody-antigen-basic/#bonus-3-dissecting-the-interface-energetics-prot-on-analysis):
#@markdown the crystal structure and two docking models of it, to be scanned one after another.
import io, urllib.request, zipfile

TUTORIAL = {
    "4G6M-matched.pdb": ("https://www.bonvinlab.org/education/HADDOCK24/HADDOCK24-antibody-antigen-basic/input-data.zip",
                         "HADDOCK24-antibody-antigen-basic/4G6M-matched.pdb", "crystal structure"),
    "cluster5_1.pdb": ("https://wenmr.science.uu.nl/haddock2.4/run/4242424242/184871-4G6M-Ab-Ag-NMR-Ab-no-passive/cluster5_1.pdb",
                       None, "best model of the top-ranked cluster"),
    "cluster2_1.pdb": ("https://wenmr.science.uu.nl/haddock2.4/run/4242424242/184871-4G6M-Ab-Ag-NMR-Ab-no-passive/cluster2_1.pdb",
                       None, "best model of the cluster with the lowest ligand-RMSD"),
}

if download_tutorial_structures:
    folder = REPO / "notebook_inputs" / "haddock_tutorial"
    folder.mkdir(parents=True, exist_ok=True)
    archives = {}
    for name, (url, member, description) in TUTORIAL.items():
        target = folder / name
        if not target.is_file():
            if member is None:
                with urllib.request.urlopen(url, timeout=120) as response:
                    target.write_bytes(response.read())
            else:
                if url not in archives:
                    with urllib.request.urlopen(url, timeout=120) as response:
                        archives[url] = zipfile.ZipFile(io.BytesIO(response.read()))
                target.write_bytes(archives[url].read(member))
        print(f"{target.relative_to(REPO).as_posix():52s} {description}")
    print("\nCopy one of these paths into pdb_file in the cell below; the antibody is chain A in all three.")
else:
    print("Tick the box to download the structures of the HADDOCK antibody-antigen tutorial.")

In [ ]:
#@title Structure and chain selection { display-mode: "form" }
pdb_file = "example-input/complex.pdb" #@param {type:"string"}
upload_pdb = False #@param {type:"boolean"}
#@markdown A PDB file with exactly two chains: the path of one, its web address, or tick `upload_pdb`
#@markdown to choose a file from your computer. Running the cell prints the chains it found and
#@markdown offers them in a dropdown; pick the one to scan there.
import re
import urllib.request

os.chdir(REPO)
INPUTS = REPO / "notebook_inputs"


def _upload(what):
    """Ask for one file in Colab and return (name, content)."""
    if not IN_COLAB:
        raise RuntimeError(f"Uploading works only in Google Colab. Type the path of {what} instead.")
    from google.colab import files
    print(f"Choose {what}:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError(f"No file was uploaded for {what}.")
    name, content = next(iter(uploaded.items()))
    for leftover in uploaded:                 # Colab also writes the files into the working folder
        Path(leftover).unlink(missing_ok=True)
    return name, content


def _safe_stem(name):
    stem = re.sub(r"[^A-Za-z0-9_-]+", "_", Path(name).stem).strip("_")
    return stem or "complex"


# --- structure -------------------------------------------------------------
if upload_pdb:
    name, content = _upload("your PDB file")
    INPUTS.mkdir(exist_ok=True)
    PDB_PATH = INPUTS / (_safe_stem(name) + ".pdb")
    PDB_PATH.write_bytes(content)
elif re.match(r"https?://", pdb_file.strip()):
    url = pdb_file.strip()
    INPUTS.mkdir(exist_ok=True)
    PDB_PATH = INPUTS / (_safe_stem(url.split("?")[0].rstrip("/").split("/")[-1]) + ".pdb")
    print("Downloading", url)
    with urllib.request.urlopen(url, timeout=180) as response:
        PDB_PATH.write_bytes(response.read())
else:
    source = Path(pdb_file.strip()).expanduser()
    if not source.is_file():
        raise FileNotFoundError(f"PDB file not found: {pdb_file}")
    source = source.resolve()
    inside_repo = REPO in source.parents
    relative = source.relative_to(REPO).as_posix() if inside_repo else ""
    if inside_repo and re.fullmatch(r"[A-Za-z0-9_./-]+\.pdb", relative):
        PDB_PATH = source
    else:                                     # PROT-ON needs a simple file name ending in .pdb
        INPUTS.mkdir(exist_ok=True)
        PDB_PATH = INPUTS / (_safe_stem(source.name) + ".pdb")
        shutil.copyfile(source, PDB_PATH)
PDB_NAME = PDB_PATH.stem

# --- the chains of the file, offered as a selection ------------------------
residues = {}
inserted = set()
with open(PDB_PATH) as handle:
    for line in handle:
        if line[:4] == "ATOM":
            residues.setdefault(line[21], set()).add(line[22:26])
            if line[26].strip():                  # column 27 is the insertion code
                inserted.add(f"{line[22:26].strip()}{line[26]} of chain {line[21]}")
if inserted:
    # PROT-ON reads columns 23-26 only, so 100 and 100A would be scanned as one position.
    # Kabat and Chothia numbering use insertion codes routinely, so refuse rather than guess.
    raise ValueError(f"{PDB_PATH.name} has residues with insertion codes "
                     f"({', '.join(sorted(inserted)[:4])}"
                     f"{' and others' if len(inserted) > 4 else ''}). PROT-ON ignores the "
                     "insertion code, so these would be scanned as one position. Renumber "
                     "the structure sequentially first, for example with  pdb_reres.")
if not residues:
    raise ValueError(f"{PDB_PATH.name} holds no ATOM records. PROT-ON needs a structure in the PDB format; "
                     "an mmCIF file has to be converted first.")


def _chain_label(chain):
    return f"{chain if chain.strip() else '(blank)'} ({len(residues[chain])} residues)"


CHAIN_OPTIONS = list(residues)
chain_summary = ", ".join(_chain_label(chain) for chain in CHAIN_OPTIONS)
if len(CHAIN_OPTIONS) != 2:
    raise ValueError(f"PROT-ON works only with dimers, but {PDB_PATH.name} has {len(CHAIN_OPTIONS)} chains: "
                     f"{chain_summary}. Please keep only the two chains of interest in the file.")
STARTS_ON = {(REPO / "example-input" / "complex.pdb").resolve(): "D"}   # the chain the
chain_ID = STARTS_ON.get(PDB_PATH.resolve(), CHAIN_OPTIONS[0])         # published example used
if chain_ID not in CHAIN_OPTIONS:
    chain_ID = CHAIN_OPTIONS[0]

CHAIN_SELECTOR = None
try:                                          # a dropdown of the chains found in the file
    import ipywidgets
    from IPython.display import display
    CHAIN_SELECTOR = ipywidgets.Dropdown(options=[(_chain_label(c), c) for c in CHAIN_OPTIONS],
                                         value=chain_ID, description="Chain to scan:",
                                         style={"description_width": "initial"})
except ImportError:                           # a plain Python session: chain_ID decides
    pass


def chain_to_scan():
    """The chain picked in the dropdown, or chain_ID when there is no dropdown."""
    chain = CHAIN_SELECTOR.value if CHAIN_SELECTOR is not None else chain_ID
    if chain not in residues:
        raise ValueError(f"Chain '{chain}' is not in {PDB_PATH.name}. Its chains are: {chain_summary}.")
    loaded_pssm = globals().get("pssm")       # only once the next cell has read one
    if loaded_pssm is not None and len(loaded_pssm) != len(residues[chain]):
        raise ValueError(f"The PSSM has {len(loaded_pssm)} rows, but chain {chain} has "
                         f"{len(residues[chain])} residues. It needs one row per residue of the chain you scan.")
    return chain


print(f"Structure : {PDB_PATH.name} with chains {chain_summary}")
if CHAIN_SELECTOR is not None:
    print("\nChoose the chain to scan below, then run the next cell.")
    display(CHAIN_SELECTOR)
else:
    print(f"\nipywidgets is not installed, so there is no dropdown and chain {chain_ID} will be scanned. "
          f'To scan the other one, run  chain_ID = "{next(c for c in CHAIN_OPTIONS if c != chain_ID)}"  '
          "in a cell of your own before step 3.")

In [ ]:
#@title PROT-ON parameters { display-mode: "form" }
cut_off = 5.0 #@param {type:"number"}
IQR = 1.5 #@param {type:"number"}
algorithm = "EvoEF1" #@param ["EvoEF1", "FoldX"]
pssm_file = "" #@param {type:"string"}
upload_pssm = False #@param {type:"boolean"}
#@markdown `cut_off`: two residues are at the interface when any of their atoms are this close (Å).
#@markdown `IQR`: how far outside the interquartile range a score has to fall to count as an outlier.
#@markdown `algorithm`: EvoEF1 is bundled; FoldX needs your own licensed copy and is
#@markdown experimental here. The PSSM file is optional.
import pandas as pd

AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
if cut_off <= 0 or IQR <= 0:
    raise ValueError("cut_off and IQR must be positive numbers.")

# --- optional PSSM ---------------------------------------------------------
PSSM_PATH, pssm = None, None
if upload_pssm or pssm_file.strip():
    if upload_pssm:
        INPUTS.mkdir(exist_ok=True)
        name, content = _upload("the PSSM file (.csv)")
        PSSM_PATH = INPUTS / "uploaded_pssm.csv"
        PSSM_PATH.write_bytes(content)
    else:
        PSSM_PATH = Path(pssm_file.strip()).expanduser().resolve()
        if not PSSM_PATH.is_file():
            raise FileNotFoundError(f"PSSM file not found: {pssm_file}")
    pssm = pd.read_csv(PSSM_PATH, sep=",")
    missing = [aa for aa in AMINO_ACIDS if aa not in pssm.columns]
    if missing:
        raise ValueError("The PSSM file must be comma separated with one column per amino acid. "
                         f"Missing columns: {', '.join(missing)}")

# --- optional FoldX --------------------------------------------------------
if algorithm == "FoldX":
    for filename in ("foldx", "rotabase.txt"):
        target = REPO / "src" / filename
        if not target.is_file():
            if not IN_COLAB:
                raise FileNotFoundError(f"To use FoldX, place '{filename}' in {REPO / 'src'}")
            _, content = _upload(f"your FoldX file '{filename}'")
            target.write_bytes(content)
    (REPO / "src" / "foldx").chmod(0o755)

print(f"Scanning : chain {chain_to_scan()} of {PDB_PATH.name} with {algorithm} "
      f"(cut-off {cut_off} Å, IQR {IQR})")
print(f"PSSM     : {PSSM_PATH.name if PSSM_PATH else 'not used'}")

## 3. Run PROT-ON
Builds every interface mutant, scores it, and runs the statistical analysis. The line below the cell shows the progress; when it finishes it prints the results folder and offers a zip of it.

In [ ]:
#@title Run PROT-ON { display-mode: "form" }
import signal, tempfile, threading, time

os.chdir(REPO)
chain_ID = chain_to_scan()                    # whatever is selected in the dropdown of step 2
results_root = REPO / "results"
prefix = f"{PDB_NAME}_chain_{chain_ID}"
output_name = f"{prefix}_{algorithm}_output"


def _work_folders():
    return {p for p in REPO.iterdir() if p.is_dir() and re.fullmatch(r"[0-9a-f]{32}", p.name)}


def _count_lines(path):
    with open(path) as handle:
        return sum(1 for line in handle if line.strip())


results_before = {p.name for p in results_root.iterdir()}
work_before = _work_folders()
root_copy = REPO / PDB_PATH.name              # proton.py leaves a copy of the input here
had_root_copy = root_copy.exists()
pssm_copy = None
if PSSM_PATH is not None:                     # the PSSM filter looks for this file name next to the scripts
    pssm_copy = REPO / "src" / f"{prefix}_pssm.csv"
    shutil.copyfile(PSSM_PATH, pssm_copy)

# PROT-ON starts its scripts with "python"; make sure that is this notebook's Python.
shim = Path(tempfile.mkdtemp(prefix="proton_python_"))
(shim / "python").write_text(f'#!/bin/sh\nexec "{sys.executable}" "$@"\n')
(shim / "python").chmod(0o755)
env = dict(os.environ, PYTHONUNBUFFERED="1", PATH=f"{shim}{os.pathsep}{os.environ.get('PATH', '')}")

command = [sys.executable, "proton.py", "--pdb", PDB_PATH.relative_to(REPO).as_posix(),
           "--chain_ID", chain_ID, "--algorithm", algorithm, "--cut_off", str(cut_off), "--IQR", str(IQR)]
log_path = REPO / "proton_run.log"
progress = {"stage": "Starting", "pass": 0, "built": 0, "contacts": 0}
STAGES = {"Defining the Interface": "Defining the interface",
          "Energies are being": "Computing binding and stability energies",
          "Outlier Detection Process": "Analysing the scores",
          "Box plot analysis": "Drawing the figures"}


def _follow(stream):
    with open(log_path, "w") as log:
        for line in stream:
            log.write(line)
            if "Distance is" in line:
                progress["contacts"] += 1
                continue
            built = re.match(r"EvoEF Building Mutation Model (\d+)", line)
            if built:
                number = int(built.group(1))
                if number == 1:
                    progress["pass"] += 1
                progress.update(stage="Building the mutant models", built=number)
                continue
            for marker, stage in STAGES.items():
                if line.startswith(marker):
                    progress["stage"] = stage


def _status(work, total, started):
    stage = progress["stage"]
    detail = ""
    scored = 0
    if work is not None:
        scored = len(list(work.glob("*_output/mutation_models/*.pdb"))) + len(list(work.glob("src/mutation_models/*.pdb")))
    of_total = f" of {total}" if total else ""
    if stage == "Building the mutant models":
        detail = f": {progress['built']}{of_total} ({'complex' if progress['pass'] < 2 else 'scanned chain alone'})"
    elif scored:
        if stage in ("Starting", "Defining the interface"):
            stage = "Building and scoring the mutants"
        detail = f": {scored}{of_total} mutations scored"
    minutes, seconds = divmod(int(time.time() - started), 60)
    return f"{stage}{detail} | {minutes:02d}:{seconds:02d} elapsed"


started = time.time()
process = subprocess.Popen(command, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, errors="replace", start_new_session=True)
reader = threading.Thread(target=_follow, args=(process.stdout,), daemon=True)
reader.start()
RESULT_DIR = None
try:
    work, total = None, None
    while process.poll() is None:
        time.sleep(1)
        if work is None:
            work = next(iter(_work_folders() - work_before), None)
        if work is not None and total is None:
            for candidate in (work / "src" / f"{prefix}_mutation_list", work / "EvoEF1" / f"{prefix}_mutation_list",
                              work / "EvoEF1" / "individual_list.txt"):
                if candidate.is_file():
                    total = _count_lines(candidate) or None
                    break
        print("\r" + _status(work, total, started).ljust(100), end="", flush=True)
    reader.join(timeout=10)
    print()
    new_results = [p for p in results_root.iterdir() if p.is_dir() and p.name not in results_before]
    finished = [p / output_name for p in new_results if (p / output_name / f"{prefix}_proton_scores").is_file()]
    if not finished:
        if progress["contacts"] == 0:
            print(f"No atoms of the two chains are within {cut_off} Å of each other, so there is no interface "
                  "to scan. Check the structure, or increase cut_off.\n")
        with open(log_path) as log:
            tail = [line.rstrip() for line in log if "Distance is" not in line][-25:]
        print("\n".join(tail))
        raise RuntimeError(f"PROT-ON did not finish. The last lines of its log are shown above; the full log is in {log_path}")
    RESULT_DIR = finished[0]
    shutil.move(str(log_path), str(RESULT_DIR / "proton_run.log"))
finally:
    if process.poll() is None:                # interrupted: stop PROT-ON and everything it started
        os.killpg(process.pid, signal.SIGTERM)
        process.wait()
    for folder in _work_folders() - work_before:
        shutil.rmtree(folder, ignore_errors=True)
    if pssm_copy is not None:
        pssm_copy.unlink(missing_ok=True)
    if not had_root_copy:
        root_copy.unlink(missing_ok=True)
    shutil.rmtree(shim, ignore_errors=True)

TABLE_NAMES = ("proton_scores", "enriching_mutations", "depleting_mutations",
               "stabilizing_enriching_mutations", "stabilizing_depleting_mutations",
               "pssm_enriching", "pssm_depleting")
RUN_LABEL = f"{PDB_NAME} chain {chain_ID}"
RUNS[RUN_LABEL] = {
    "result_dir": RESULT_DIR, "prefix": prefix, "algorithm": algorithm, "ddg": f"DDG_{algorithm}_Scores",
    "cut_off": cut_off, "IQR": IQR,
    "tables": {name: (pd.read_csv(RESULT_DIR / f"{prefix}_{name}", sep=" ")
                      if (RESULT_DIR / f"{prefix}_{name}").is_file() else None) for name in TABLE_NAMES},
}

tables = RUNS[RUN_LABEL]["tables"]
scores = tables["proton_scores"]
ddg = RUNS[RUN_LABEL]["ddg"]


def download_results(include_mutation_models=True):
    """Pack this run's results folder into a zip file; in Colab the download then starts."""
    import zipfile
    run_folder = RESULT_DIR.parent
    archive = run_folder.with_suffix(".zip")
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(RESULT_DIR.rglob("*")):
            if path.is_file() and (include_mutation_models or "mutation_models" not in path.parts):
                bundle.write(path, Path(run_folder.name) / path.relative_to(RESULT_DIR))
    print(f"Saved {archive} ({archive.stat().st_size / 1e6:.1f} MB)")
    if IN_COLAB:
        from google.colab import files
        files.download(str(archive))
    return archive


minutes, seconds = divmod(int(time.time() - started), 60)
print(f"PROT-ON finished in {minutes} min {seconds} s.")
print("Results folder:", RESULT_DIR)

try:                                          # the zip holds the tables, the figures, the log and the mutants
    import ipywidgets
    from IPython.display import display
    with_models = ipywidgets.Checkbox(value=True, description="include the mutant structures", indent=False)
    button = ipywidgets.Button(description="Download the results", icon="download")
    log = ipywidgets.Output()

    def _download(_):
        button.disabled = True
        with log:
            download_results(with_models.value)
        button.disabled = False

    button.on_click(_download)
    print("\nDownload the results of this run?")
    display(ipywidgets.HBox([button, with_models]), log)
except ImportError:
    print("\nTo save the results as a zip file, run  download_results()  in a cell of your own.")

## 4. Results

**The two ends of the scan in 3D.** The cell below draws the mutant PROT-ON predicts to bind best next to the one it predicts to bind worst. Drag to rotate, scroll to zoom. These are EvoEF1's own models, the ones its scores were computed from, so this is a look at what was scored rather than a new model.

In [ ]:
#@title The best and the worst mutant in 3D { display-mode: "form" }
designer_mutations_only = True #@param {type:"boolean"}
surface_opacity = 0.65 #@param {type:"slider", min:0.2, max:1.0, step:0.05}
#@markdown With `designer_mutations_only` ticked, the two mutants are taken from the designer tables; untick it to
#@markdown take the lowest and the highest ΔΔG of the whole scan, outliers or not.
import py3Dmol
from IPython.display import Markdown, display

MODELS = RESULT_DIR / "mutation_models"
INTERFACE = []
interface_list = RESULT_DIR / f"{prefix}_interface_aa_list"
if interface_list.is_file():
    for line in interface_list.read_text().splitlines():
        parts = line.split()
        if len(parts) == 3 and parts[2].lstrip("-").isdigit():
            INTERFACE.append(int(parts[2]))

PICKED_FROM = {
    "stabilizing_enriching_mutations": "enriching designer mutation (an outlier that also stabilizes the chain)",
    "enriching_mutations": "enriching outlier (none of them stabilize the chain)",
    "stabilizing_depleting_mutations": "depleting designer mutation (an outlier that also stabilizes the chain)",
    "depleting_mutations": "depleting outlier (none of them stabilize the chain)",
    "proton_scores": "mutation of the scan, outlier or not",
}


def _strongest(names, lowest):
    """The strongest mutation of the first of these tables that holds one, else of the whole scan."""
    for name in names:
        table = RUNS[RUN_LABEL]["tables"].get(name)
        if table is not None and not table.empty:
            return table.loc[table[ddg].idxmin() if lowest else table[ddg].idxmax()], name
    return scores.loc[scores[ddg].idxmin() if lowest else scores[ddg].idxmax()], "proton_scores"


def _model_of(row):
    """The mutant structure EvoEF1 or FoldX built for this mutation."""
    position = row["Positions"]
    code = f"{position[0]}{chain_ID}{position[1:]}{row['Mutations']}"
    found = sorted(MODELS.glob(f"*{code}.pdb")) if MODELS.is_dir() else []
    return found[0] if found else None


def _atoms(text):
    return [line for line in text.splitlines() if line[:4] == "ATOM" and len(line) > 54]


def _centre(text, number):
    """Where to hang the label: the CA of the mutated residue, or any of its atoms."""
    for wanted in ("CA", ""):
        for line in _atoms(text):
            if line[21] == chain_ID and line[22:26].strip() == str(number) and line[12:16].strip().startswith(wanted):
                return {"x": float(line[30:38]), "y": float(line[38:46]), "z": float(line[46:54])}
    return None


def _xyz(text, chain, number=None):
    picked = []
    for line in _atoms(text):
        if line[21] != chain or (number is not None and line[22:26].strip() != str(number)):
            continue
        picked.append((line[22:26].strip(), float(line[30:38]), float(line[38:46]), float(line[46:54])))
    return picked


def _touched(text, partner, number):
    """The residues of the partner chain the mutated side chain reaches, at PROT-ON's own cut-off."""
    limit = RUNS[RUN_LABEL]["cut_off"] ** 2
    mutated = _xyz(text, chain_ID, number)
    return sorted({int(resi) for resi, x, y, z in _xyz(text, partner)
                   if any((x - mx) ** 2 + (y - my) ** 2 + (z - mz) ** 2 <= limit for _, mx, my, mz in mutated)})


enriching = ["stabilizing_enriching_mutations", "enriching_mutations"] if designer_mutations_only else []
depleting = ["stabilizing_depleting_mutations", "depleting_mutations"] if designer_mutations_only else []
panels = [(*_strongest(enriching, True), "Best", "green"),
          (*_strongest(depleting, False), "Worst", "red")]

view = py3Dmol.view(width=900, height=420, viewergrid=(1, 2), linked=False)
rows, missing = [], []
for column, (row, source, title, colour) in enumerate(panels):
    mutation = f"{row['Positions']}{row['Mutations']}"
    path = _model_of(row)
    reached = "no model was kept"
    if path is None:
        missing.append(mutation)
    else:
        text = path.read_text()
        cell = (0, column)
        number = int(row["Positions"][1:])
        partner = next((c for c in sorted({line[21] for line in _atoms(text)}) if c != chain_ID), chain_ID)
        touched = _touched(text, partner, number)
        reached = f"{len(touched)} residues of chain {partner}"
        elsewhere = [resi for resi in sorted({int(r) for r, *_ in _xyz(text, partner)}) if resi not in set(touched)]
        view.addModel(text, "pdb", viewer=cell)
        view.setStyle({"chain": partner}, {"cartoon": {"color": "white"}}, viewer=cell)
        view.addSurface(py3Dmol.VDW, {"opacity": surface_opacity, "color": "lightgrey"},
                        {"chain": partner, "resi": elsewhere}, viewer=cell)
        if touched:                           # the patch of the partner this very side chain reaches
            view.addSurface(py3Dmol.VDW, {"opacity": min(1.0, surface_opacity + 0.25), "color": "gold"},
                            {"chain": partner, "resi": touched}, viewer=cell)
        view.setStyle({"chain": chain_ID}, {"cartoon": {"color": "lightsteelblue"}}, viewer=cell)
        if INTERFACE:
            view.addStyle({"chain": chain_ID, "resi": INTERFACE},
                          {"stick": {"radius": 0.1, "color": "steelblue"}}, viewer=cell)
        view.addStyle({"chain": chain_ID, "resi": number}, {"stick": {"radius": 0.3, "color": colour}}, viewer=cell)
        spot = _centre(text, number)
        if spot is not None:
            view.addLabel(f"{mutation}   ΔΔG {row[ddg]:+.2f}",
                          {"position": spot, "backgroundColor": colour, "backgroundOpacity": 0.85,
                           "fontColor": "white", "fontSize": 12}, viewer=cell)
        view.zoomTo(viewer=cell)

    rows.append(f"| **{title}** | {mutation} | {row[ddg]:+.2f} | {row['DDG_Stability_Scores']:+.2f} | "
                f"strongest {PICKED_FROM[source]} | {reached} |")

display(Markdown("\n".join(["| | mutation | ΔΔG binding | ΔΔG stability | picked as the | side chain reaches |",
                            "|---|---|---|---|---|---|"] + rows)))
if missing:
    print("No mutant structure was kept for: " + ", ".join(missing))
print(f"Left: the best mutant, right: the worst. The partner chain is the surface, gold where this side chain "
      f"touches it; chain {chain_ID} is the ribbon, its interface residues drawn as thin sticks.")
view.show()

In [ ]:
#@title Heatmap of every scanned mutation (hover to read the values) { display-mode: "form" }
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Blue-grey-red, from the validated diverging pair. Equal step count per arm.
BLUE_ARM = ["#0d366b", "#184f95", "#256abf", "#2a78d6", "#5598e7", "#9ec5f4", "#cde2fb"]
RED_ARM = ["#fde2de", "#f7b7b0", "#ef8a84", "#e34948", "#c62f33", "#9e2227", "#73161b"]
NEUTRAL = "#f0efec"
ENRICH, DEPLETE, BULK = "#2a78d6", "#e34948", "#9a9a94"
INK, INK_2, LINE = "#1b1b19", "#55554f", "#dcdcd6"

# Grouped by chemistry, so neighbouring columns are comparable.
CLASSES = [("aliphatic", list("GAVLIMP")), ("aromatic", list("FWY")),
           ("polar", list("STCNQ")), ("basic", list("KRH")), ("acidic", list("DE"))]
FONT = dict(family="Helvetica, Arial, sans-serif", size=13, color=INK)


def _position_number(label):
    return int(re.search(r"-?\d+", label).group())


def _ordered_positions(scores):
    return sorted(scores["Positions"].unique(), key=_position_number)


WHISKER = RUNS[RUN_LABEL]["IQR"]        # the multiplier this run called its outliers with


def _fences(values):
    """PROT-ON's own outlier rule: Q1 - k*IQR and Q3 + k*IQR, with this run's own k."""
    q1, q3 = np.quantile(values, 0.25), np.quantile(values, 0.75)
    return q1 - WHISKER * (q3 - q1), q3 + WHISKER * (q3 - q1)


def _diverging(limit):
    stops = []
    for index, colour in enumerate(BLUE_ARM):
        stops.append((0.5 * index / len(BLUE_ARM), colour))
    stops.append((0.5, NEUTRAL))
    for index, colour in enumerate(RED_ARM):
        stops.append((0.5 + 0.5 * (index + 1) / len(RED_ARM), colour))
    return [[round(position, 4), colour] for position, colour in stops]


def _base(figure, title, subtitle, height, width=820):
    figure.update_layout(
        template="plotly_white", font=FONT, width=width, height=height,
        margin=dict(l=90, r=40, t=86 if subtitle else 64, b=64),
        title=dict(text=f"<b>{title}</b>" + (f"<br><span style='font-size:12px;color:{INK_2}'>"
                                             f"{subtitle}</span>" if subtitle else ""),
                   x=0, xanchor="left", font=dict(size=16)),
        plot_bgcolor="white", paper_bgcolor="white", showlegend=False)
    figure.update_xaxes(showgrid=False, zeroline=False, linecolor=LINE, ticks="outside",
                        tickcolor=LINE, title_font=dict(size=12, color=INK_2))
    figure.update_yaxes(showgrid=False, zeroline=False, linecolor=LINE, ticks="outside",
                        tickcolor=LINE, title_font=dict(size=12, color=INK_2))
    return figure


def heatmap(scores, ddg, chain_id, name, clip_percentile=99.0):
    """Every scanned mutation, position by substitution.

    The colour limit is symmetric about zero and robust; the subtitle gives it, and cells
    beyond it carry a dot rather than being silently flattened.
    """
    positions = _ordered_positions(scores)
    columns, boundaries = [], []
    for index, (_, members) in enumerate(CLASSES):
        columns.extend(members)
        if index < len(CLASSES) - 1:
            boundaries.append(len(columns) - 0.5 + len(boundaries))
            columns.append(" " * (index + 1))        # a unique, invisible spacer column

    residues = {residue for _, members in CLASSES for residue in members}
    table = scores.pivot_table(index="Positions", columns="Mutations", values=ddg)
    table = table.reindex(index=positions, columns=columns)
    limit = float(np.percentile(np.abs(scores[ddg]), clip_percentile))

    # One cell of every row is the residue already there, so it has no score. Left empty it
    # reads as "no effect" on a scale whose middle is near-white; grey it out instead.
    blank = [[1 if (column in residues and pd.isna(table.loc[position, column])) else None
              for column in columns] for position in positions]
    figure = go.Figure(go.Heatmap(
        z=blank, x=columns, y=positions, showscale=False, hoverinfo="skip",
        colorscale=[[0, "#e6e5e0"], [1, "#e6e5e0"]], xgap=1, ygap=1))
    figure.add_trace(go.Heatmap(
        z=table.values, x=columns, y=positions, zmid=0, zmin=-limit, zmax=limit,
        colorscale=_diverging(limit), xgap=1, ygap=1,
        hovertemplate="%{y}%{x} &nbsp; ΔΔG %{z:.2f} kcal/mol<extra></extra>",
        colorbar=dict(title=dict(text="ΔΔG<br>kcal/mol", font=dict(size=11)),
                      thickness=12, len=0.55, outlinewidth=0, tickfont=dict(size=11),
                      ticks="outside", y=0.5)))

    wild_x, wild_y, over_x, over_y, over_text = [], [], [], [], []
    for position in positions:
        wild = position[0]
        if wild in columns:
            wild_x.append(wild)
            wild_y.append(position)
        for mutation in columns:
            value = table.loc[position, mutation] if mutation in table.columns else np.nan
            if pd.notna(value) and abs(value) > limit:
                over_x.append(mutation)
                over_y.append(position)
                over_text.append(f"{position}{mutation} ΔΔG {value:.2f}")
    figure.add_trace(go.Scatter(
        x=wild_x, y=wild_y, mode="markers", hoverinfo="text",
        text=[f"{p} is the wild-type residue" for p in wild_y],
        marker=dict(symbol="x-thin", size=7, line=dict(color="#b8b8b2", width=1.4))))
    if over_x:
        figure.add_trace(go.Scatter(
            x=over_x, y=over_y, mode="markers", hoverinfo="text", text=over_text,
            marker=dict(symbol="circle", size=5.5, color="#15150f",
                        line=dict(color="white", width=1.1))))

    for boundary in boundaries:
        figure.add_vline(x=boundary, line_color="white", line_width=6)
    annotations = []
    cursor = 0
    for label, members in CLASSES:
        centre = cursor + (len(members) - 1) / 2
        annotations.append(dict(x=centre, y=1.0, yref="paper", yanchor="bottom",
                                yshift=24, text=label, showarrow=False,
                                font=dict(size=11, color=INK_2)))
        cursor += len(members) + 1
    beyond = len(over_x)
    figure.update_layout(annotations=annotations)
    _base(figure, f"Chain {chain_id} of {name}: every scanned mutation",
          f"Blue enriches binding, red depletes it, white is no change. Colour saturates at "
          f"±{limit:.1f} kcal/mol; {beyond} cell{'s' if beyond != 1 else ''} beyond that "
          f"carry a dot. ✕ marks the wild-type residue.",
          height=max(430, 21 * len(positions) + 210), width=880)
    figure.update_xaxes(side="top", tickfont=dict(size=12), title_text="")
    figure.update_yaxes(autorange="reversed", tickfont=dict(size=11), title_text="",
                        range=[len(positions) - 0.5, -0.5])
    figure.update_layout(margin=dict(l=90, r=40, t=132, b=30))
    return figure


heatmap(scores, ddg, chain_ID, PDB_NAME).show()

own = ([] if RESULT_DIR is None else
       sorted(RESULT_DIR.glob(f"{prefix}_*.png")) + sorted(RESULT_DIR.glob(f"{prefix}_*.svg")))
if own:
    print(f"PROT-ON draws its own figures as well: {len(own)} PNG and SVG files in "
          f"{RESULT_DIR.name}. Those are the ones the web server and the paper show; the "
          f"figures in this notebook are drawn from the same table.")

In [ ]:
#@title The scan position by position, and where the outlier rule cuts { display-mode: "form" }
#@markdown Every position's spread, the distribution with PROT-ON's fences drawn on it,
#@markdown and the two filters that pick the designer mutations.
def _strip(scores, ddg, group, order, title, subtitle, axis_title, height):
    """One row per group: every mutation as a point, the median as a tick."""
    low, high = _fences(scores[ddg])
    figure = go.Figure()
    figure.add_vrect(x0=low, x1=high, fillcolor="#f7f7f4", line_width=0, layer="below")
    for value, name in ((low, "lower fence"), (high, "upper fence")):
        figure.add_vline(x=value, line_color="#c4c4bd", line_width=1, line_dash="dot",
                         annotation_text=f"{name} {value:.2f}", annotation_position="top",
                         annotation_font=dict(size=10, color=INK_2))
    figure.add_vline(x=0, line_color="#8c8c85", line_width=1)

    rows = {name: frame for name, frame in scores.groupby(group)}
    rng = np.random.default_rng(0)
    for index, name in enumerate(order):
        frame = rows.get(name)
        if frame is None or frame.empty:
            continue
        values = frame[ddg].to_numpy()
        colours = np.where(values <= low, ENRICH, np.where(values >= high, DEPLETE, BULK))
        sizes = np.where((values <= low) | (values >= high), 7.0, 4.6)
        jitter = rng.uniform(-0.17, 0.17, len(values))
        figure.add_trace(go.Scatter(
            x=values, y=index + jitter, mode="markers",
            marker=dict(color=colours, size=sizes, opacity=0.85,
                        line=dict(width=0.6, color="white")),
            hovertemplate=[f"{p}{m} &nbsp; ΔΔG {v:.2f} kcal/mol<extra></extra>"
                           for p, m, v in zip(frame["Positions"], frame["Mutations"], values)]))
        median = float(np.median(values))
        figure.add_trace(go.Scatter(
            x=[median], y=[index], mode="markers",
            marker=dict(symbol="line-ns", size=15, line=dict(color=INK, width=2)),
            hovertemplate=f"median {median:.2f} over {len(values)} mutations<extra></extra>"))

    _base(figure, title, subtitle, height=height)
    figure.update_yaxes(tickmode="array", tickvals=list(range(len(order))), ticktext=order,
                        range=[len(order) - 0.2, -0.8], title_text="",
                        tickfont=dict(size=11))
    figure.update_xaxes(title_text=axis_title, zeroline=False)
    return figure


def per_position(scores, ddg, chain_id, name):
    order = _ordered_positions(scores)
    low, high = _fences(scores[ddg])
    outliers = int(((scores[ddg] <= low) | (scores[ddg] >= high)).sum())
    return _strip(
        scores, ddg, "Positions", order,
        f"Chain {chain_id} of {name}: the scan position by position",
        f"Each point is one substitution, the tick is the position's median.<br>"
        f"{outliers} of {len(scores)} mutations fall outside PROT-ON's fences "
        f"(blue enriching, red depleting); the shaded band is the ordinary range.",
        "ΔΔG of binding (kcal/mol)", height=max(430, 20 * len(order) + 190))


def distribution(scores, ddg, chain_id, name):
    """Where the outlier rule cuts, drawn on the distribution it cuts."""
    values = scores[ddg].to_numpy()
    low, high = _fences(values)
    # The fences go into the bin edges: a bar straddling one would be coloured wholly
    # inlier or wholly outlier and would then disagree with the counts annotated beside it.
    edges = np.unique(np.concatenate([np.histogram_bin_edges(values, bins=46), [low, high]]))
    counts, _ = np.histogram(values, bins=edges)
    figure = go.Figure()
    for count, lower, upper in zip(counts, edges[:-1], edges[1:]):
        if not count:
            continue
        centre = (lower + upper) / 2
        outlier = centre <= low or centre >= high
        colour = ENRICH if centre <= low else DEPLETE if centre >= high else BULK
        figure.add_trace(go.Bar(
            x=[centre], y=[int(count)], width=(upper - lower) * 0.92,
            marker=dict(color=colour, line_width=0), opacity=0.9,
            text=[str(int(count))] if outlier else None,   # the tails are a few pixels tall
            textposition="outside", textfont=dict(size=9, color=colour), cliponaxis=False,
            hovertemplate=f"{lower:.2f} to {upper:.2f} kcal/mol<br>"
                          f"{int(count)} mutations<extra></extra>"))
    for value in (low, high):
        figure.add_vline(x=value, line_color="#6f6f68", line_width=1, line_dash="dot")
    figure.add_vline(x=0, line_color="#8c8c85", line_width=1)
    enriching = int((values <= low).sum())
    depleting = int((values >= high).sum())
    top = int(counts.max())
    for value, text, colour in ((low, f"{enriching} enriching", ENRICH),
                                (high, f"{depleting} depleting", DEPLETE)):
        figure.add_annotation(x=value, y=top * 0.97, text=text, showarrow=False,
                              xanchor="right" if value < 0 else "left",
                              xshift=-6 if value < 0 else 6,
                              font=dict(size=11, color=colour))
    _base(figure, f"Chain {chain_id} of {name}: how the outlier rule cuts the scan",
          f"All {len(values)} substitutions. The dotted lines are Q1 − {WHISKER}·IQR = {low:.2f} "
          f"and Q3 + {WHISKER}·IQR = {high:.2f}; everything beyond them is reported as an outlier.",
          height=420)
    figure.update_xaxes(title_text="ΔΔG of binding (kcal/mol)")
    figure.update_yaxes(title_text="mutations", gridcolor="#eeeeea", showgrid=True)
    return figure


def selection_map(scores, ddg, stability, chain_id, name):
    """Both filters on one pair of axes: the binding fences and the stability cut.

    It shows how far the designer mutations sit from the cut, whether the two energies
    are related at all for this interface, and what the stability filter removed.
    """
    low, high = _fences(scores[ddg])
    binding, stable = scores[ddg].to_numpy(), scores[stability].to_numpy()
    outlier = (binding <= low) | (binding >= high)
    designer = outlier & (stable < 0)

    figure = go.Figure()
    # Shade only what the tool keeps: an outlier in binding that is also stabilizing.
    floor = min(0.0, float(min(stable))) * 1.1 - 0.2   # stays below zero even if nothing
                                                       # in this scan is stabilizing
    span = float(max(binding)) - float(min(binding))
    for x0, x1, colour in ((float(min(binding)) - 0.1 * span, low, "#eaf1fb"),
                           (high, float(max(binding)) + 0.1 * span, "#fbecec")):
        figure.add_shape(type="rect", x0=x0, x1=x1, y0=floor, y1=0, layer="below",
                         fillcolor=colour, line_width=0)
    figure.add_vline(x=0, line_color="#e2e2dc", line_width=1)
    for value in (low, high):
        figure.add_vline(x=value, line_color="#6f6f68", line_width=1, line_dash="dot")
    figure.add_hline(y=0, line_color="#6f6f68", line_width=1, line_dash="dot")

    label = [f"{p}{m}" for p, m in zip(scores["Positions"], scores["Mutations"])]
    groups = [
        (~outlier, BULK, 4.4, 0.45, "circle", "inside the fences"),
        (outlier & ~designer, "#6f6f68", 7.5, 0.95, "circle-open", "outlier, not stabilizing"),
        (designer & (binding < 0), ENRICH, 9.5, 0.95, "circle", "designer, enriching"),
        (designer & (binding > 0), DEPLETE, 9.5, 0.95, "circle", "designer, depleting")]
    for mask, colour, size, opacity, symbol, legend in groups:
        if not mask.any():
            continue
        figure.add_trace(go.Scatter(
            x=binding[mask], y=stable[mask], mode="markers", name=legend,
            marker=dict(color=colour, size=size, opacity=opacity, symbol=symbol,
                        line=dict(width=0.8, color="white" if "open" not in symbol else colour)),
            hovertemplate=[f"<b>{t}</b><br>binding %{{x:.2f}}<br>stability %{{y:.2f}}"
                           f"<extra>{legend}</extra>" for t in np.array(label)[mask]]))
    # One label per position, on its strongest designer mutation.
    positions = scores["Positions"].to_numpy()
    strongest = {}
    for index in np.flatnonzero(designer):
        current = strongest.get(positions[index])
        if current is None or abs(binding[index]) > abs(binding[current]):
            strongest[positions[index]] = index
    # A few labels only, and always the enriching side: ranking by |binding| alone would
    # drop the one enriching designer mutation behind a row of depleting ones.
    def _pick(indices, count):
        return sorted(indices, key=lambda i: -abs(binding[i]))[:count]

    enriching_side = [i for i in strongest.values() if binding[i] < 0]
    depleting_side = [i for i in strongest.values() if binding[i] >= 0]
    labelled = _pick(enriching_side, 1) + _pick(depleting_side, 3)
    offsets = (-40, -12, 18, 46)
    for rank, index in enumerate(sorted(labelled, key=lambda i: -stable[i])):
        position = positions[index]
        siblings = int((designer & (positions == position)).sum()) - 1
        text = label[index] + (f" (+{siblings} more)" if siblings else "")
        figure.add_annotation(x=binding[index], y=stable[index], text=text,
                              showarrow=True, arrowhead=0, arrowwidth=0.8,
                              arrowcolor="#9a9a94",
                              ax=52 if binding[index] > 0 else -46,
                              ay=offsets[rank % len(offsets)],
                              xanchor="left" if binding[index] > 0 else "right",
                              font=dict(size=10, color=INK))

    kept = int(designer.sum())
    lost = int((outlier & ~designer).sum())
    # ranks then Pearson: Spearman without pulling scipy into the notebook
    rho = pd.Series(binding).rank().corr(pd.Series(stable).rank())
    _base(figure, f"Chain {chain_id} of {name}: what the two filters keep",
          f"The shaded corners are what PROT-ON keeps: an outlier in binding that also "
          f"stabilizes the chain.<br>{kept} of {len(binding)} mutations clear both cuts; "
          f"{lost} are binding outliers the stability filter removes. A label counts the "
          f"other designer mutations at the same position. Rank correlation between the "
          f"two energies: {rho:+.2f}.", height=580, width=800)
    figure.update_xaxes(title_text="ΔΔG of binding (kcal/mol) → depleting")
    figure.update_yaxes(title_text="ΔΔG of stability (kcal/mol) → destabilizing",
                        gridcolor="#eeeeea", showgrid=True)
    figure.update_layout(showlegend=True,
                         legend=dict(orientation="h", y=-0.16, x=0, font=dict(size=11),
                                     bgcolor="rgba(0,0,0,0)"))
    return figure


per_position(scores, ddg, chain_ID, PDB_NAME).show()
distribution(scores, ddg, chain_ID, PDB_NAME).show()

stability = next((column for column in scores.columns
                  if column.startswith("DDG_") and "Stability" in column), None)
if stability is None:
    print("no stability column in this scan, so the selection map is skipped")
else:
    selection_map(scores, ddg, stability, chain_ID, PDB_NAME).show()

**Reading the tables.** `DDG_<algorithm>_Scores` is the binding score of the mutant minus that of the wild type: negative values are predicted to enrich binding, positive values to deplete it. `DDG_Stability_Scores` is the same difference for the stability of the scanned chain alone: negative values are stabilizing. The *designer mutations* are the enriching and depleting outliers that are also stabilizing.

In [ ]:
#@title Designer mutations { display-mode: "form" }
from IPython.display import Image, Markdown, display

if RESULT_DIR is None:
    raise RuntimeError("Run step 3 first.")

display(Markdown(
    f"**{len(scores)} mutations** were scanned at **{scores['Positions'].nunique()} interface positions** "
    f"of chain {chain_ID}. Binding-enriching outliers: **{len(tables['enriching_mutations'])}** "
    f"({len(tables['stabilizing_enriching_mutations'])} also stabilizing). "
    f"Binding-depleting outliers: **{len(tables['depleting_mutations'])}** "
    f"({len(tables['stabilizing_depleting_mutations'])} also stabilizing)."))

titles = {"stabilizing_enriching_mutations": "Designer mutations: binding enriching and stabilizing",
          "stabilizing_depleting_mutations": "Designer mutations: binding depleting and stabilizing",
          "pssm_enriching": "Enriching designer mutations that also pass the PSSM filter",
          "pssm_depleting": "Depleting designer mutations that also pass the PSSM filter"}
for name, title in titles.items():
    table = tables[name]
    if table is None:
        continue
    display(Markdown(f"#### {title}"))
    if table.empty:
        print("None found.")
    else:
        display(table.sort_values(ddg).reset_index(drop=True))

## 5. Experimental: re-score the designer mutations with HADDOCK3

Does a second energy function agree with EvoEF1 about which mutations matter? [HADDOCK3](https://github.com/haddocking/haddock3) can say. `pip install haddock3` compiles it from source — a few minutes on Colab — and carries a statically linked CNS. HADDOCK3 prints the terms at every start and they are worth repeating: CNS is free for non-profit use, and a commercial user is responsible for holding a licence.

**The structures are the scan's own.** EvoEF1 and FoldX each build every mutant they score, so HADDOCK3 re-scores those coordinates rather than making its own. Nothing is rebuilt: this is two energy functions reading one structure, and it is deterministic, so each mutation is scored exactly once. The wild type is the structure you gave the scan; EvoEF1 repairs it before mutating, so the wild type is the one model here that has not been through that step, and the energy minimisation before scoring is what puts them on the same footing. The shortlist goes through one HADDOCK3 run as an ensemble, so the whole thing is a single run.

**What is compared.** For a complex of exactly two chains, `per_interface_scoring` returns the score of the whole refined complex, not of the interface alone. So **`WT_minus_mutant` is not an interface-only quantity** — it moves with whatever the mutation does inside the chain it sits in, which for a side-chain substitution is not small. The table shows the terms behind each score — van der Waals, electrostatics and buried surface area — so you can see which of them a mutation actually moved.

**Mind the sign.** The HADDOCK score is better when more negative, so a mutation that improves the complex makes *wild type − mutant* positive — the opposite way round from `DDG_<algorithm>_Scores`. Two methods that agree therefore show a **negative** rank correlation.

A HADDOCK score is not a binding free energy and these differences are not ΔΔG values; they rank mutations, they do not measure them.

In [ ]:
#@title Install HADDOCK3 { display-mode: "form" }
install_haddock3 = False #@param {type:"boolean"}
#@markdown A few minutes: there is no wheel, so pip compiles haddock3 from source. The
#@markdown source distribution carries a statically linked CNS, so nothing else is needed.
if install_haddock3:
    import tempfile

    if shutil.which("haddock3") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "haddock3"], check=True)
    out = subprocess.run(["haddock3", "--version"], capture_output=True, text=True)
    print(out.stdout.strip() or out.stderr.strip())
    from haddock.core.defaults import cns_exec
    print("CNS:", cns_exec, "| present:", Path(str(cns_exec)).is_file())

    # haddock3 pins plotly 7; step 1 pinned plotly 5 with kaleido 0.2.1, which is what
    # PROT-ON's own figures are written with. Left alone, the upgrade silently stops every
    # png and svg from being written -- proton.py ignores the plotting step's exit code, so
    # the run still reports success and the figure cells simply show nothing.
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "plotly==5.24.1", "kaleido==0.2.1"], check=True)
    with tempfile.TemporaryDirectory() as tmp:
        probe = subprocess.run(
            [sys.executable, "-c", "import sys, plotly, plotly.graph_objects as go; "
             f"go.Figure().write_image({str(Path(tmp) / 'probe.png')!r}); "
             "print(plotly.__version__)"],
            capture_output=True, text=True)
    if probe.returncode == 0:
        print("figure export: working, plotly", probe.stdout.strip(),
              "(haddock3's own plots are not used here)")
    else:
        print("figure export: BROKEN -",
              (probe.stderr.strip().splitlines() or ["no error text"])[-1])
else:
    print("Tick the box to install HADDOCK3. Step 5 is optional; steps 1-4 do not need it.")

In [ ]:
#@title Re-score with HADDOCK3 { display-mode: "form" }
which_mutations = "designer" #@param ["designer", "enriching outliers", "depleting outliers"]
max_mutations = 12 #@param {type:"integer"}
max_per_position = 2 #@param {type:"integer"}
em_steps = 200 #@param {type:"integer"}
ncores = 2 #@param {type:"integer"}
show_every_score = True #@param {type:"boolean"}
#@markdown `max_mutations + 1` structures get refined. The cell prints the time per model,
#@markdown so you can judge what a full scan would cost. `max_per_position` stops the
#@markdown shortlist filling up with one position's worth of near-identical mutations.
import re, signal, threading, time

import pandas as pd
from IPython.display import Markdown, display

def _is_hydrogen(line):
    element = line[76:78].strip()
    return element == "H" if element else line[12:16].strip().lstrip("0123456789").startswith("H")


def _structure_for(row, models_dir, chain):
    """The mutant structure the scan itself built and scored, if it is still on disk."""
    position = row["Positions"]
    code = f"{position[0]}{chain}{position[1:]}{row['Mutations']}"
    found = sorted(models_dir.glob(f"*{code}.pdb"))
    return found[0] if found else None


def _ensemble(path, structures):
    """One MODEL per structure, the wild type first, hydrogens left to CNS."""
    with open(path, "w") as handle:
        for index, source in enumerate(structures, start=1):
            handle.write(f"MODEL     {index:>4}\n")
            kept = 0
            for line in Path(source).read_text().splitlines():
                if line[:4] == "ATOM" and not _is_hydrogen(line):
                    handle.write(line.rstrip() + "\n")
                    kept += 1
            if not kept:
                raise RuntimeError(f"{source} holds no ATOM records")
            handle.write("ENDMDL\n")
        handle.write("END\n")
    return path


def _shortlist(table, column, limit, per_position):
    """Which mutations to re-score: strongest first, both signs, spread over positions.

    Ranking by |DDG| alone hands the list to whichever sign is more numerous, and often
    to one position, over a range too narrow for the two methods to disagree about.
    """
    ranked = table.sort_values(column, key=lambda values: values.abs(), ascending=False)
    enriching = [index for index, value in ranked[column].items() if value < 0]
    depleting = [index for index, value in ranked[column].items() if value >= 0]
    taken, order = {}, []
    while (enriching or depleting) and len(order) < limit:
        progressed = False
        for side in (enriching, depleting):
            while side and len(order) < limit:
                index = side.pop(0)
                position = ranked.loc[index, "Positions"]
                if taken.get(position, 0) >= per_position:
                    continue
                taken[position] = taken.get(position, 0) + 1
                order.append(index)
                progressed = True
                break
        if not progressed:
            break
    return ranked.loc[order]


def _interface_scores(step_dir, first, second):
    """The HADDOCK score of each refined model, read from emscoring's own table.

    For two chains, emscoring_<A>_<B>.tsv and the overall emscoring.tsv carry the same
    number: CNS only recomputes per-pair energies for three chains or more
    (print_coorheader.cns, the `ncomponents eq 2` branch).
    """
    wanted = {frozenset((first, second))}
    found = []
    for tsv in sorted(Path(step_dir).glob("emscoring_*.tsv")):
        chains = tsv.stem.split("_")[1:]
        found.append(tsv.name)
        if len(chains) == 2 and frozenset(chains) in wanted:
            return pd.read_csv(tsv, sep="\t")
    raise FileNotFoundError(
        f"no per-interface score for chains {first} and {second} in {step_dir}. "
        f"Files there: {found or sorted(p.name for p in Path(step_dir).glob('*.tsv'))}. "
        "If only emscoring.tsv is present, per_interface_scoring did not take effect.")


def _components(step_dir, structure, first, second):
    """The HADDOCK score of one scored model and the terms it is made of.

    emscoring's table carries the score alone; the terms are in the scored structure's own
    REMARK lines, which haddock3 parses for us. For a chain pair it reports the HADDOCK
    score, the van der Waals and electrostatic energies and the buried surface area.
    """
    from haddock.gear.haddockmodel import HaddockModel

    path = Path(step_dir) / Path(structure).name
    model = HaddockModel(path)
    wanted = (f"{first}_{second}", f"{second}_{first}")
    terms = next((model.interface_energies[key] for key in wanted
                  if key in model.interface_energies), None)
    if terms is None:
        raise RuntimeError(f"{path.name} carries no interface energies for chains "
                           f"{first} and {second}; per_interface_scoring did not take effect")
    return {"HADDOCK_score": terms["HADDOCKscore"], "vdw": terms["vdw"],
            "elec": terms["elec"], "bsa": terms["bsa"]}


def _model_index(table):
    """Which model of the ensemble a scored structure came from: topoaa splits it into
    <name>_1.pdb, <name>_2.pdb and so on, so the last number in the name is the one."""
    names = table["original_name"].astype(str)
    index = names.str.extract(r"(\d+)(?!.*\d)")[0]
    if index.isna().any():
        raise RuntimeError(f"cannot tell which ensemble model these came from: "
                           f"{names[index.isna()].tolist()[:3]}")
    return index.astype(int)


def _run_haddock(work, name):
    """Run one workflow, showing which module is working and how far it has got."""
    state = {"module": "starting", "total": 0, "done": 0}
    log_path = work / f"{name}.log"
    process = subprocess.Popen(["haddock3", f"{name}.toml"], cwd=work, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, errors="replace",
                               start_new_session=True)

    def follow(stream):
        with open(log_path, "w") as log:
            for line in stream:
                log.write(line)
                log.flush()
                module = re.search(r"\] \[(\w+)\]", line)
                if module and module.group(1) != state["module"]:
                    state.update(module=module.group(1), total=0, done=0)
                jobs = re.search(r"Running CNS Jobs n=(\d+)", line)
                if jobs:
                    state["total"] = int(jobs.group(1))
                finished = re.search(r"(\d+) tasks finished", line)
                if finished:
                    state["done"] = int(finished.group(1))

    reader = threading.Thread(target=follow, args=(process.stdout,), daemon=True)
    reader.start()
    begun = time.time()
    try:
        while process.poll() is None:
            time.sleep(1)
            minutes, seconds = divmod(int(time.time() - begun), 60)
            of = f" {state['done']}/{state['total']} jobs" if state["total"] else ""
            print(f"\r  {state['module']}{of} | {minutes:02d}:{seconds:02d} elapsed".ljust(68),
                  end="", flush=True)
        reader.join(timeout=5)
        print()
    finally:
        if process.poll() is None:            # interrupted: stop CNS and everything it started
            os.killpg(process.pid, signal.SIGTERM)
            process.wait()
    return process.returncode


ready = True
if RESULT_DIR is None:
    print("Step 5 needs a finished PROT-ON run: go back to step 3 first.")
    ready = False
elif shutil.which("haddock3") is None:
    print("Step 5 skipped: HADDOCK3 is not installed. Tick the box in the cell above, run it "
          "(a few minutes), then run this cell. Steps 1-4 are complete without it.")
    ready = False

if ready:
    source = {"designer": ["stabilizing_enriching_mutations", "stabilizing_depleting_mutations"],
              "enriching outliers": ["enriching_mutations"],
              "depleting outliers": ["depleting_mutations"]}[which_mutations]
    frames = [tables[name] for name in source
              if tables.get(name) is not None and not tables[name].empty]
    if not frames:
        raise RuntimeError(f"this run found no {which_mutations}; try another set")

    # ignore_index matters: "designer" concatenates two tables, both numbered from 0.
    picked = _shortlist(pd.concat(frames, ignore_index=True), ddg, max_mutations,
                        max_per_position)
    if picked.empty:
        raise RuntimeError(f"this run found no {which_mutations}; try another set")

    chosen = [(int(row["Positions"][1:]), row["Mutations"]) for _, row in picked.iterrows()]
    enriching_n = int((picked[ddg] < 0).sum())
    print(f"re-scoring {len(chosen)} mutations "
          f"({enriching_n} binding-enriching, {len(chosen) - enriching_n} depleting, "
          f"at most {max_per_position} per position):",
          ", ".join(f"{p}{m}" for p, m in zip(picked['Positions'], picked['Mutations'])))

    models_dir = RESULT_DIR / "mutation_models"
    if not models_dir.is_dir():
        raise RuntimeError(f"this run kept no mutant structures ({models_dir} is missing), "
                           "so there is nothing for HADDOCK3 to re-score")
    found = [(row, _structure_for(row, models_dir, chain_ID)) for _, row in picked.iterrows()]
    absent = [f"{row['Positions']}{row['Mutations']}" for row, path in found if path is None]
    if absent:
        raise RuntimeError(f"no built structure for {', '.join(absent)} in {models_dir}. "
                           "Re-run step 3 with the mutant models kept.")

    wild_type = PDB_PATH                  # the structure the scan was given

    work = REPO / "haddock3_rescoring"
    shutil.rmtree(work, ignore_errors=True)
    work.mkdir(parents=True)
    _ensemble(work / "complex_ensemble.pdb", [wild_type] + [path for _, path in found])
    print(f"scoring structures built by {algorithm}: {wild_type.name} and "
          f"{len(found)} mutant models from {models_dir.name}")

    # One run, scoring the whole ensemble: every mutation is refined and scored exactly
    # once. The structures are fixed, so running it again would only repeat the answer.
    (work / "complex.toml").write_text(f"""
run_dir = "complex_run"
mode = "local"
ncores = {ncores}
molecules = ["complex_ensemble.pdb"]

[topoaa]
iniseed = 917

[emref]
nemsteps = {em_steps}
iniseed = 917
tolerance = 20

[emscoring]
per_interface_scoring = true
""".lstrip())

    partner_chain = next((c for c in CHAIN_OPTIONS if c != chain_ID), None)
    if partner_chain is None:
        raise RuntimeError("this structure has only one chain, so it has no interface to score")

    LABEL = {1: "wild type"}
    LABEL.update({i + 2: f"{p}{m}" for i, (p, m) in enumerate(chosen)})

    started = time.time()
    print(f"\ncomplex ensemble ({len(chosen) + 1} structures)")
    if _run_haddock(work, "complex") != 0:
        with open(work / "complex.log") as log:
            tail = [line.rstrip() for line in log if line.strip()][-25:]
        print("\n".join(tail))
        raise RuntimeError(f"HADDOCK3 failed. Its last lines are above; the full log is "
                           f"{work / 'complex.log'}")
    step = next(d for d in sorted((work / "complex_run").iterdir()) if "emscoring" in d.name)
    scored = _interface_scores(step, chain_ID, partner_chain)
    scored["model"] = _model_index(scored)
    models = len(chosen) + 1
    elapsed = time.time() - started
    print(f"\n{models} models refined and scored in {elapsed / 60:.1f} min "
          f"({elapsed / models:.1f} s per model)")
    print(f"this scan found {len(scores)} mutations; at this rate re-scoring all of them "
          f"would take {len(scores) * elapsed / models / 3600:.1f} h")

    scored["structure_scored"] = scored["model"].map(LABEL)
    terms = pd.DataFrame([_components(step, row["structure"], chain_ID, partner_chain)
                          for _, row in scored.iterrows()], index=scored.index)
    scored = pd.concat([scored, terms], axis=1).sort_values("model").reset_index(drop=True)

    if 1 not in set(scored["model"]):
        raise RuntimeError("the wild type (model 1) is missing from the scores; the model "
                           "numbering did not survive the run, so the differences below "
                           "cannot be trusted")
    wt_score = float(scored.loc[scored["model"] == 1, "HADDOCK_score"].iloc[0])

    ddg_of = {LABEL[i + 2]: value for i, value in enumerate(picked[ddg].to_numpy())}
    result = scored[["model", "structure_scored"] + (["original_name"] if show_every_score
                                                     else [])].copy()
    result[f"DDG_{algorithm}"] = result["structure_scored"].map(ddg_of)
    for column in ("HADDOCK_score", "vdw", "elec", "bsa"):
        result[column] = scored[column]
    result["WT_minus_mutant"] = wt_score - result["HADDOCK_score"]
    result.loc[result["model"] == 1, "WT_minus_mutant"] = float("nan")

    display(Markdown("#### The wild type and each mutant, scored by HADDOCK3"))
    print(f"HADDOCK_score is the total; vdw, elec and bsa are the terms behind it. "
          f"WT_minus_mutant is positive when the mutant complex scores better, which is the "
          f"opposite sign to DDG_{algorithm}.")
    display(result.round(2))

    if result["HADDOCK_score"].nunique() < len(result) / 2:
        print("warning: several structures share a score, which usually means models were "
              "mapped to the wrong mutation rather than that they are genuinely equal. "
              "Compare the model numbers against the structures they came from.")

    print("\nThe HADDOCK column is the whole refined complex, not the interface alone, so it "
          "carries each chain's internal energy too (see the text above the cell).")

    pair = result[[f"DDG_{algorithm}", "WT_minus_mutant"]].dropna()
    if len(pair) > 2:
        rho = pair.rank().corr().iloc[0, 1]
        print(f"  rank correlation with DDG_{algorithm} over {len(pair)} mutations: "
              f"{rho:+.2f} ({'agreement' if rho < 0 else 'disagreement'})")